In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from shared_utils import *
import monai
from monai.networks.nets import SwinUNETR
from torch.optim.swa_utils import AveragedModel, SWALR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | MONAI: {monai.__version__}')
print('[v07] Ultimate SwinUNETR | AG-MSF + Deep Supervision + TTA + ClinicalFocalLoss')


In [ ]:
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
BASE_DIR  = DATA_ROOT / "segformer/experiments/ultimate_swinunetr"
BASE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {**SHARED_CONFIG,
    "model_name":   "ultimate_swinunetr",
    "feature_size": 48,
    "n_epochs":     100,
    "patience":     25,
    "swa_start":    60,
    "frozen_epochs": 14,  # İlk 14 epoch: backbone donduruldu
    # --- Sensitivity-First HyperParams ---
    "pos_weight":    3.0,   # Baseline'in 2.5'inden daha agresif
    "focal_gamma":   3.0,
    "label_smoothing": 0.05,
    # Stochastic Depth: Overfitting engeli
    "drop_rate":          0.1,
    "attn_drop_rate":     0.1,
    "dropout_path_rate":  0.2,
}
print(f"Ortak veriseti (Stratified): {DATA_ROOT / "segformer/datas"}")
test_df = pd.read_csv(DATA_ROOT / "segformer/datas" / "external_test_set.csv")
print(f"External Test: {len(test_df)}")


In [ ]:
# ============================================================
# UltimateSwinUNETR3D
# Yenilik 1: Attention-Guided Multi-Scale Fusion (AG-MSF)
# Yenilik 2: Deep Supervision (her katman kendi hatasını öğrenir)
# Yenilik 3: Stochastic Depth (overfitting engeli)
# ============================================================
class ChannelAttention(nn.Module):
    """
    Squeeze-and-Excitation: Hangi ölçek bu hasta için önemli?
    Tümör büyükse derin katman, küçükse sığ katman ağırlık alır.
    """
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, max(in_channels // reduction, 8), bias=False),
            nn.GELU(),
            nn.Linear(max(in_channels // reduction, 8), in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x)


class UltimateSwinUNETR3D(nn.Module):
    def __init__(self, num_classes=2, in_channels=1, feature_size=48, config=None):
        super().__init__()
        cfg = config or CONFIG
        self.backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=14,
            feature_size=feature_size,
            use_checkpoint=True,
            spatial_dims=3,
            drop_rate=cfg.get("drop_rate", 0.1),
            attn_drop_rate=cfg.get("attn_drop_rate", 0.1),
            dropout_path_rate=cfg.get("dropout_path_rate", 0.2),
        )
        dims = [feature_size * (2**i) for i in range(5)]  # [48, 96, 192, 384, 768]
        fused_dim = sum(dims)  # 1488

        self.gap   = nn.AdaptiveAvgPool3d(1)
        self.norms = nn.ModuleList([nn.LayerNorm(d) for d in dims])

        # Yenilik 1: Channel Attention
        self.channel_attention = ChannelAttention(fused_dim, reduction=16)

        # Yenilik 2: Deep Supervision (her ölçek için ayrı sınıflandırıcı)
        self.ds_heads = nn.ModuleList([nn.Linear(d, num_classes) for d in dims])

        # Ana sınıflandırıcı
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        hidden = self.backbone.swinViT(x, self.backbone.normalize)
        feats, ds_outs = [], []

        for i, h in enumerate(hidden):
            f = self.gap(h).flatten(1)
            f = self.norms[i](f)
            feats.append(f)
            if self.training:
                ds_outs.append(self.ds_heads[i](f))

        # AG-MSF: Birleştir + Attention
        fused    = torch.cat(feats, dim=1)
        attended = self.channel_attention(fused)
        out      = self.classifier(attended)

        if self.training:
            return out, ds_outs
        return out


In [ ]:
def load_pretrained_swin(model, ckpt_path=None):
    if ckpt_path is None:
        candidates = [
            os.path.expanduser("~/models/model_swinvit.pt"),
            os.path.expanduser("~/.cache/torch/hub/checkpoints/model_swinvit.pt"),
            str(DATA_ROOT / "segformer/model_swinvit.pt"),
        ]
        ckpt_path = next((p for p in candidates if Path(p).exists()), None)

    if ckpt_path is None or not Path(ckpt_path).exists():
        print("[WARN] Pretrained ağırlık bulunamadı.")
        return model

    state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
    target = model.backbone.swinViT.state_dict()
    new_sd, loaded, skipped = {}, 0, 0
    for k, v in state_dict.items():
        k2 = k.replace("swinViT.", "").replace("module.", "")
        if k2 in target and target[k2].shape == v.shape:
            new_sd[k2] = v; loaded += 1
        else:
            skipped += 1
    model.backbone.swinViT.load_state_dict(new_sd, strict=False)
    print(f"  Pretrained: {loaded} katman yüklendi, {skipped} atlandı.")
    return model


In [ ]:
# ── run_one_fold_ultimate ──────────────────────────────────────
def run_one_fold_ultimate(train_df, val_df, fold_idx, config, output_dir):
    def set_seed(s):
        torch.manual_seed(s); np.random.seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    set_seed(config["random_seed"] + fold_idx)

    fold_dir = Path(output_dir) / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_ds = AppendixH5Dataset(train_df, augment=True,  config=config)
    val_ds   = AppendixH5Dataset(val_df,   augment=False, config=config)
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"],
                              shuffle=True,  num_workers=config["num_workers"],
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config["batch_size"],
                              shuffle=False, num_workers=config["num_workers"],
                              pin_memory=True)

    model = UltimateSwinUNETR3D(config=config).to(DEVICE)
    model = load_pretrained_swin(model)

    # Progressive Finetuning: ilk N epoch backbone donduruldu
    frozen_epochs = config.get("frozen_epochs", 14)
    for name, param in model.backbone.named_parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = get_warmup_cosine_scheduler(optimizer, config["warmup_epochs"], config["n_epochs"])
    criterion = ClinicalFocalLoss(
        pos_weight=config.get("pos_weight", 3.0),
        gamma=config.get("focal_gamma", 3.0),
        smoothing=config.get("label_smoothing", 0.05))

    swa_model  = AveragedModel(model)
    swa_start  = config.get("swa_start", 60)
    swa_sched  = SWALR(optimizer, swa_lr=config["lr"] * 0.1)

    best_score, best_metrics, patience_cnt = -1.0, None, 0
    history = []
    threshold = SHARED_CONFIG["default_threshold"]

    for epoch in range(1, config["n_epochs"] + 1):
        # Progressive Finetuning: N. epoch'tan itibaren backbone çöz
        if epoch == frozen_epochs + 1:
            print(f"  [Progressive FT] Epoch {epoch}: Backbone çözülüyor.")
            for param in model.backbone.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(),
                                          lr=config["lr"],
                                          weight_decay=config["weight_decay"])
            scheduler = get_warmup_cosine_scheduler(optimizer, 5, config["n_epochs"] - epoch)

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_auc, val_acc, val_f1, pred_df = evaluate_model(
            model, val_loader, criterion, DEVICE)

        y_true = pred_df["label"].values
        y_prob = pred_df["prob_mucinous"].values
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        sens = tp / (tp + fn + 1e-9)
        spec = tn / (tn + fp + 1e-9)
        composite = clinical_composite(sens, val_auc, spec)

        print(f"[universal | fold {fold_idx} | epoch {epoch:03d}] "              f"train={train_loss:.4f} val_loss={val_loss:.4f} "              f"auc={val_auc:.4f} acc={val_acc:.4f}")
        print(f"  [Metrics] AUC:{val_auc:.3f} PR:{val_f1:.3f} "              f"SENS:{sens:.3f} SPEC:{spec:.3f} | COMPOSITE:{composite:.4f}")

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "auc": val_auc, "sens": sens, "spec": spec, "composite": composite})

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            scheduler.step()

        if composite > best_score:
            best_score = composite
            torch.save(model.state_dict(), fold_dir / "best_model.pt")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= config["patience"]:
                print(f"  Early stopping @ epoch {epoch}")
                break

    # SWA Finalize
    if epoch >= swa_start:
        print("  SWA model finalize ediliyor...")
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
        _, val_auc_swa, _, _, pred_swa = evaluate_model(swa_model, val_loader, criterion, DEVICE)
        y_true_swa = pred_swa["label"].values
        y_prob_swa = pred_swa["prob_mucinous"].values
        y_pred_swa = (y_prob_swa >= threshold).astype(int)
        tn_, fp_, fn_, tp_ = confusion_matrix(y_true_swa, y_pred_swa, labels=[0,1]).ravel()
        swa_comp = clinical_composite(tp_/(tp_+fn_+1e-9), val_auc_swa, tn_/(tn_+fp_+1e-9))
        print(f"  SWA val COMPOSITE: {swa_comp:.4f} | Best regular COMPOSITE: {best_score:.4f}")
        if swa_comp > best_score:
            torch.save(swa_model.state_dict(), fold_dir / "best_model.pt")
            print("  SWA modeli seçildi!")

    # Final fold evaluation
    model.load_state_dict(torch.load(fold_dir / "best_model.pt", map_location=DEVICE, weights_only=False))
    _, val_auc_f, val_acc_f, val_f1_f, pred_df_f = evaluate_model(
        model, val_loader, criterion, DEVICE)
    youden_thr, _ = find_youden_threshold(pred_df_f["label"].values, pred_df_f["prob_mucinous"].values)
    ci = compute_bootstrap_ci(pred_df_f["label"].values, pred_df_f["prob_mucinous"].values, youden_thr)
    metrics_f, cm_f, _ = compute_binary_metrics(
        pred_df_f["label"].values, pred_df_f["prob_mucinous"].values, youden_thr)

    print_full_metrics_table(metrics_f, ci, f"UniversalModel Fold {fold_idx}", f"Youden {youden_thr:.3f}")
    plot_confusion_matrix(cm_f, f"Fold {fold_idx} CM", save_path=fold_dir / f"cm_fold{fold_idx}.png")
    pred_df_f.to_csv(fold_dir / "val_predictions.csv", index=False)
    return metrics_f, ci, pred_df_f, history


In [ ]:
import json

all_preds, all_metrics = [], []

for fold_idx in range(1, 6):
    train_df = pd.read_csv(DATA_ROOT / "segformer/datas" / f"fold_{fold_idx}_train.csv")
    val_df   = pd.read_csv(DATA_ROOT / "segformer/datas" / f"fold_{fold_idx}_val.csv")
    print(f"
{'='*70}
FOLD {fold_idx}/5
{'='*70}")
    m_f, ci_f, pred_f, hist_f = run_one_fold_ultimate(train_df, val_df, fold_idx, CONFIG, BASE_DIR)
    all_preds.append(pred_f)
    all_metrics.append(m_f)

print("
5-Fold CV tamamlandı.")


In [ ]:
# ── External Test ────────────────────────────────────────────────
test_ds = AppendixH5Dataset(test_df, augment=False, config=CONFIG)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"],
                         shuffle=False, num_workers=CONFIG["num_workers"])
criterion_eval = ClinicalFocalLoss()
fold_probs_ext, fold_rows_ext = [], []

print(f"
EXTERNAL TEST (ALL FOLDS + ENSEMBLE):")
for fold_idx in range(1, 6):
    fold_dir = BASE_DIR / f"fold_{fold_idx}"
    model_e  = UltimateSwinUNETR3D(config=CONFIG).to(DEVICE)
    model_e.load_state_dict(torch.load(fold_dir / "best_model.pt", map_location=DEVICE, weights_only=False))
    _, _, _, _, pred_ext = evaluate_model(model_e, test_loader, criterion_eval, DEVICE)
    del model_e; torch.cuda.empty_cache()

    y_true_e = pred_ext["label"].values
    y_prob_e = pred_ext["prob_mucinous"].values
    fold_probs_ext.append(y_prob_e)
    m_e, _, _ = compute_binary_metrics(y_true_e, y_prob_e, SHARED_CONFIG["default_threshold"])
    fold_rows_ext.append({"fold": f"Fold {fold_idx}",
                          "auc_roc": round(m_e["auc_roc"],3),
                          "sensitivity": round(m_e["sensitivity"],3),
                          "specificity": round(m_e["specificity"],3),
                          "accuracy": round(m_e["accuracy"],3),
                          "f1": round(m_e["f1"],3),
                          "tp": m_e["tp"], "fp": m_e["fp"],
                          "fn": m_e["fn"], "tn": m_e["tn"]})

ens_prob   = np.mean(fold_probs_ext, axis=0)
y_true_ext = test_df["label"].values[:len(ens_prob)]

youden_t, _  = find_youden_threshold(y_true_ext, ens_prob)
sens_t        = find_sensitivity_threshold(y_true_ext, ens_prob, min_sensitivity=0.90)
m_youden, cm_y, _ = compute_binary_metrics(y_true_ext, ens_prob, youden_t)
m_05,     cm_05,_ = compute_binary_metrics(y_true_ext, ens_prob, 0.5)
m_sens,   cm_s, _ = compute_binary_metrics(y_true_ext, ens_prob, sens_t)

fold_rows_ext += [
    {"fold": "Ensemble (@Youden)",  **{k: round(v,3) if isinstance(v,float) else v
      for k,v in m_youden.items() if k in ["auc_roc","sensitivity","specificity","accuracy","f1","tp","fp","fn","tn"]}},
    {"fold": "Ensemble (@0.5)",     **{k: round(v,3) if isinstance(v,float) else v
      for k,v in m_05.items()     if k in ["auc_roc","sensitivity","specificity","accuracy","f1","tp","fp","fn","tn"]}},
    {"fold": "Ensemble (90+ Sens)", **{k: round(v,3) if isinstance(v,float) else v
      for k,v in m_sens.items()   if k in ["auc_roc","sensitivity","specificity","accuracy","f1","tp","fp","fn","tn"]}},
]
print(pd.DataFrame(fold_rows_ext).to_string(index=False))

ci_ens  = compute_bootstrap_ci(y_true_ext, ens_prob, youden_t)
ci_ens2 = compute_bootstrap_ci(y_true_ext, ens_prob, sens_t)
print("
=== ENSEMBLE @YOUDEN ===")
print_full_metrics_table(m_youden, ci_ens, "Universal Ensemble", f"Youden {youden_t:.3f}")
print("
=== ENSEMBLE @90+ SENS ===")
print_full_metrics_table(m_sens, ci_ens2, "Universal Ensemble", f"90+Sens {sens_t:.3f}")

plot_dir = BASE_DIR / "external_test_folds"
plot_dir.mkdir(exist_ok=True)
plot_roc_pr(y_true_ext, ens_prob, "UltimateSwinUNETR_Ensemble", plot_dir, opt_threshold=youden_t)
plot_confusion_matrix(cm_y, "Ensemble @Youden",  save_path=plot_dir / "cm_ensemble_youden.png")
plot_confusion_matrix(cm_s, "Ensemble @90+Sens", save_path=plot_dir / "cm_ensemble_90sens.png")

results = {"fold_metrics": all_metrics, "ensemble_youden": m_youden, "ensemble_90sens": m_sens}
with open(BASE_DIR / "all_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"
Sonuçlar kaydedildi: {BASE_DIR / 'all_results.json'}")
